# Gate 4 — Visual data sanity

The automated tests prove the tensors have the right *shape*. They cannot prove the images
have the right *content*. An inverted radiograph, a lesion box landing in the wrong corner,
or an augmentation that erases the finding all produce perfectly valid float32 arrays and a
training curve that converges — and a model that has learned nothing useful.

In medical imaging this step is not optional. Look at every plot below before training.

Run from the repo root:
```
.venv\Scripts\python.exe -m jupyter lab notebooks/01_data_sanity.ipynb
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from onnm import CLASS_NAMES
from onnm.config import load_config
from onnm.dataset import build_records, build_transforms
from onnm.explainability import annotation_path_for, load_annotation, map_box_to_model_space
from onnm.io_radiograph import read_radiograph

cfg = load_config(ROOT / "configs/base.yaml")
SIZE = int(cfg.data.image_size)
records = build_records(cfg)
print(f"{len(records)} records loaded")

## 1. Raw images, before any processing

Bone should be **bright** and soft tissue **dark**. If any panel looks like a photographic
negative, the photometric handling is wrong and everything downstream is compromised.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 4, figsize=(14, 11))

for row, class_idx in enumerate(range(3)):
    pool = [r for r in records if r["label"] == class_idx]
    for col, record in enumerate(rng.choice(pool, 4, replace=False)):
        arr, meta = read_radiograph(record["image"])
        ax = axes[row, col]
        ax.imshow(arr, cmap="gray")
        ax.set_title(f"{CLASS_NAMES[class_idx]}\n{record['image_id']}  {arr.shape}", fontsize=9)
        ax.axis("off")

fig.suptitle("Raw radiographs by class — bone must appear BRIGHT", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Ground-truth lesion boxes in model space

This is the check that validates the resize-and-pad arithmetic end to end. The green box is
computed by `map_box_to_model_space`, drawn over the image the model actually receives.
**If the boxes do not sit on the lesions, the localisation metrics are fiction.**

In [ ]:
import matplotlib.patches as patches

transform = build_transforms(cfg, "val", keep_meta=True)
tumour = [r for r in records if r["label"] > 0 and annotation_path_for(r["image_id"], cfg).is_file()]

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, record in zip(axes.ravel(), rng.choice(tumour, 8, replace=False)):
    ann = load_annotation(annotation_path_for(record["image_id"], cfg))
    sample = transform(record)
    ax.imshow(sample["image"][0].numpy(), cmap="gray")

    for box in ann["boxes"]:
        x0, y0, x1, y1 = map_box_to_model_space(box, ann["height"], ann["width"], SIZE)
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                       linewidth=1.8, edgecolor="lime", facecolor="none"))
    ax.set_title(f"{CLASS_NAMES[record['label']]} · {ann['labels'][0] if ann['labels'] else ''}",
                 fontsize=9)
    ax.axis("off")

fig.suptitle("Lesion boxes mapped into 256×256 model space — must land ON the lesion", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Aspect ratio and padding

The chain resizes the longest side to 256 and pads the rest, so anatomy keeps its proportions.
Padding shows as flat black bands. A long-bone film squashed into a square would deform the
lesion margin and periosteal reaction — the morphological signs that carry the diagnosis.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for col, record in enumerate(rng.choice(records, 5, replace=False)):
    raw, _ = read_radiograph(record["image"])
    axes[0, col].imshow(raw, cmap="gray")
    axes[0, col].set_title(f"original {raw.shape}", fontsize=9)
    axes[0, col].axis("off")

    out = transform(record)["image"][0].numpy()
    axes[1, col].imshow(out, cmap="gray")
    axes[1, col].set_title(f"model input {out.shape}", fontsize=9)
    axes[1, col].axis("off")

fig.suptitle("Original (top) vs model input (bottom) — proportions must be preserved", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Augmentation

Sixteen draws of the **same** image. They should differ visibly but stay anatomically
plausible — every panel must still look like a radiograph a radiologist could be handed.
Note there is no vertical flip: an upside-down film never occurs in practice.

In [ ]:
train_transform = build_transforms(cfg, "train")
record = tumour[0]

fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(train_transform(record)["image"][0].numpy(), cmap="gray")
    ax.set_title(f"draw {i}", fontsize=8)
    ax.axis("off")

fig.suptitle(f"Augmentation draws — {record['image_id']} ({CLASS_NAMES[record['label']]})", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Intensity distributions

After percentile scaling and ImageNet normalisation the three classes should overlap heavily.
A clean separation here would be bad news: it would mean the model can pick the class from
brightness alone, which is a scanner or exposure artefact, not pathology.

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 4.5))

for class_idx, name in enumerate(CLASS_NAMES):
    pool = [r for r in records if r["label"] == class_idx]
    values = np.concatenate([
        transform(r)["image"][0].numpy().ravel()[::37]
        for r in rng.choice(pool, min(40, len(pool)), replace=False)
    ])
    ax_left.hist(values, bins=80, alpha=0.5, density=True, label=name)

ax_left.set_title("Normalised pixel intensity by class (should overlap)")
ax_left.set_xlabel("value after ImageNet normalisation")
ax_left.legend()

counts = [sum(1 for r in records if r["label"] == i) for i in range(3)]
bars = ax_right.bar(CLASS_NAMES, counts, color=["#4C78A8", "#F58518", "#E45756"])
ax_right.bar_label(bars)
ax_right.set_title(f"Class distribution — malignant is {100 * counts[2] / sum(counts):.1f}%")

plt.tight_layout()
plt.show()

## 6. Surrogate patient grouping

BTXRD ships no patient identifier, but images are clearly not independent. Each row below is
one reconstructed group. The panels should look like **the same patient from different
angles**. If they show unrelated anatomy, the grouping heuristic is over-merging and
`split.group_strategy` should be reconsidered.

In [ ]:
from collections import Counter

sizes = Counter(r["patient_id"] for r in records)
multi = [g for g, n in sizes.most_common() if 2 <= n <= 4][:4]

fig, axes = plt.subplots(len(multi), 4, figsize=(13, 3.2 * len(multi)))
for row, group in enumerate(multi):
    members = [r for r in records if r["patient_id"] == group]
    for col in range(4):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(members):
            arr, _ = read_radiograph(members[col]["image"])
            ax.imshow(arr, cmap="gray")
            ax.set_title(f"{members[col]['image_id']}\n{CLASS_NAMES[members[col]['label']]}",
                         fontsize=8)
    axes[row, 0].set_ylabel(group)

fig.suptitle("Reconstructed patient groups — each row should be ONE patient", fontsize=13)
plt.tight_layout()
plt.show()

---
## Checklist before training

- [ ] Bone is bright, soft tissue dark — no inverted images
- [ ] Lesion boxes land on lesions
- [ ] Aspect ratio preserved; padding is flat black bands
- [ ] Augmented images remain anatomically plausible
- [ ] Class intensity histograms overlap
- [ ] Each group row looks like one patient

All ticked → `python scripts/overfit_check.py`